# NHANES Pulse Consumption — Data Download

Downloads dietary recall data (Individual Foods, Day 1) and demographic files for
NHANES 1999–2018 (10 cycles). Identifies pulse-containing foods and saves
person-level pulse consumption data for analysis.

**Reference:** Drewnowski et al. (2025) *Frontiers in Nutrition* 10.3389/fnut.2025.1638519

In [1]:
import os, hashlib, re, time
import requests
import numpy as np
import pandas as pd
import pyreadstat

DATA_DIR   = os.path.abspath(os.path.join('..', 'data'))
RAW_NHANES = os.path.join(DATA_DIR, 'raw', 'nhanes')
DERIVED    = os.path.join(DATA_DIR, 'derived')
for d in [RAW_NHANES, DERIVED]:
    os.makedirs(d, exist_ok=True)

## Configuration

NHANES dietary file naming changed over cycles:
- 1999–2000: `DRXIFF.xpt` / `DRXFMT.xpt` (prefix `DRX`/`DRD`, no suffix)
- 2001–2002: `DRXIFF_B.xpt` / `DRXFMT_B.xpt`
- 2003–2004+: `DR1IFF_{suffix}.xpt` / `DRXFCD_{suffix}.xpt`

In [2]:
NHANES_BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public'
FPED_BASE = 'https://www.ars.usda.gov/ARSUserFiles/80400530/apps'
FPED_DIR = os.path.join(DATA_DIR, 'raw', 'fped')
os.makedirs(FPED_DIR, exist_ok=True)

COHORTS = {
    '1999-2000': {'year': 1999, 'suffix': '',  'diet_file': 'DRXIFF',   'fcd_file': 'DRXFMT',
                  'fped_file': None},
    '2001-2002': {'year': 2001, 'suffix': 'B', 'diet_file': 'DRXIFF_B', 'fcd_file': 'DRXFMT_B',
                  'fped_file': None},
    '2003-2004': {'year': 2003, 'suffix': 'C', 'diet_file': 'DR1IFF_C', 'fcd_file': 'DRXFCD_C',
                  'fped_file': 'MPED_0304.xls'},
    '2005-2006': {'year': 2005, 'suffix': 'D', 'diet_file': 'DR1IFF_D', 'fcd_file': 'DRXFCD_D',
                  'fped_file': 'FPED_0506.xls'},
    '2007-2008': {'year': 2007, 'suffix': 'E', 'diet_file': 'DR1IFF_E', 'fcd_file': 'DRXFCD_E',
                  'fped_file': 'FPED_0708.xls'},
    '2009-2010': {'year': 2009, 'suffix': 'F', 'diet_file': 'DR1IFF_F', 'fcd_file': 'DRXFCD_F',
                  'fped_file': 'FPED_0910.xls'},
    '2011-2012': {'year': 2011, 'suffix': 'G', 'diet_file': 'DR1IFF_G', 'fcd_file': 'DRXFCD_G',
                  'fped_file': 'FPED_1112.xls'},
    '2013-2014': {'year': 2013, 'suffix': 'H', 'diet_file': 'DR1IFF_H', 'fcd_file': 'DRXFCD_H',
                  'fped_file': 'FPED_1314.xls'},
    '2015-2016': {'year': 2015, 'suffix': 'I', 'diet_file': 'DR1IFF_I', 'fcd_file': 'DRXFCD_I',
                  'fped_file': 'FPED_1516.xls'},
    '2017-2018': {'year': 2017, 'suffix': 'J', 'diet_file': 'DR1IFF_J', 'fcd_file': 'DRXFCD_J',
                  'fped_file': 'FPED_1718.xls'},
}

## Download helpers

In [3]:
def download(url, dest, retries=4):
    """Download with caching and retry."""
    if os.path.exists(dest):
        return dest
    for attempt in range(retries):
        try:
            print(f'  GET {url}')
            r = requests.get(url, timeout=300)
            r.raise_for_status()
            with open(dest, 'wb') as f:
                f.write(r.content)
            print(f'    -> {os.path.basename(dest)} ({len(r.content):,} bytes)')
            return dest
        except Exception as e:
            wait = 2 ** (attempt + 1)
            print(f'    RETRY {attempt+1}/{retries} after {wait}s: {e}')
            time.sleep(wait)
    raise RuntimeError(f'Failed to download {url}')

def read_xpt(path):
    try:
        df, _ = pyreadstat.read_xport(path)
    except UnicodeDecodeError:
        df, _ = pyreadstat.read_xport(path, encoding='latin1')
    return df

## Download dietary and demographic files

In [4]:
raw = {}

for cycle, cfg in COHORTS.items():
    print(f'\n=== {cycle} ===')
    yr = cfg['year']
    sfx = cfg['suffix']
    cdir = os.path.join(RAW_NHANES, cycle.replace('-', '_'))
    os.makedirs(cdir, exist_ok=True)
    d = {}

    # Demographics
    demo_file = f'DEMO_{sfx}.xpt' if sfx else 'DEMO.xpt'
    demo_url = f'{NHANES_BASE}/{yr}/DataFiles/{demo_file}'
    path = download(demo_url, os.path.join(cdir, demo_file))
    d['DEMO'] = read_xpt(path)
    print(f'  DEMO: {len(d["DEMO"]):,} rows')

    # Individual Foods Day 1
    diet_file = f'{cfg["diet_file"]}.xpt'
    diet_url = f'{NHANES_BASE}/{yr}/DataFiles/{diet_file}'
    path = download(diet_url, os.path.join(cdir, diet_file))
    d['DR1IFF'] = read_xpt(path)
    print(f'  DR1IFF: {len(d["DR1IFF"]):,} rows')

    # Food code description file
    fcd_file = f'{cfg["fcd_file"]}.xpt'
    fcd_url = f'{NHANES_BASE}/{yr}/DataFiles/{fcd_file}'
    try:
        path = download(fcd_url, os.path.join(cdir, fcd_file))
        d['FCD'] = read_xpt(path)
        print(f'  FCD: {len(d["FCD"]):,} rows')
    except Exception as e:
        print(f'  FCD: not available ({e})')

    # FPED/MPED database
    if cfg['fped_file']:
        fped_path = os.path.join(FPED_DIR, cfg['fped_file'])
        if not os.path.exists(fped_path):
            fped_url = f'{FPED_BASE}/{cfg["fped_file"]}'
            download(fped_url, fped_path)
        d['FPED'] = pd.read_excel(fped_path, engine='xlrd')
        print(f'  FPED: {len(d["FPED"]):,} rows')

    raw[cycle] = d

print(f'\nDownloaded {len(raw)} cycles')


=== 1999-2000 ===


  DEMO: 9,965 rows


  DR1IFF: 127,840 rows
  FCD: 4,311 rows

=== 2001-2002 ===
  DEMO: 11,039 rows


  DR1IFF: 143,004 rows
  FCD: 6,974 rows

=== 2003-2004 ===
  DEMO: 10,122 rows


  DR1IFF: 131,164 rows
  FCD: 6,940 rows
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


...

  FPED: 8,536 rows

=== 2015-2016 ===
  DEMO: 9,971 rows


  DR1IFF: 121,481 rows
  FCD: 8,690 rows
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


  FPED: 8,690 rows

=== 2017-2018 ===
  DEMO: 9,254 rows


  DR1IFF: 112,683 rows
  FCD: 7,083 rows
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


  FPED: 7,083 rows

Downloaded 10 cycles


## Harmonize column names across cycles

The 1999–2000 cycle uses `DRD` prefix for food code and gram weight columns,
while later cycles use `DR1I` prefix.

In [5]:
# Check column names for food code and gram weight
for cycle, d in raw.items():
    df = d['DR1IFF']
    fc_cols = [c for c in df.columns if 'FDCD' in c]
    gr_cols = [c for c in df.columns if 'GRMS' in c or 'GRM' in c]
    st_cols = [c for c in df.columns if 'DRST' in c or 'DRSTZ' in c]
    print(f'{cycle}: food_code={fc_cols}, grams={gr_cols}, status={st_cols}')
    if 'FCD' in d:
        print(f'  FCD cols: {list(d["FCD"].columns)[:8]}')

1999-2000: food_code=['DRDIFDCD'], grams=['DRXIGRMS'], status=['DRDDRSTS']
  FCD cols: ['FMTNAME', 'START', 'LABEL']
2001-2002: food_code=['DRDIFDCD'], grams=['DRXIGRMS'], status=['DRDDRSTZ']
  FCD cols: ['FMTNAME', 'START', 'LABEL']
2003-2004: food_code=['DR1IFDCD'], grams=['DR1IGRMS'], status=['DR1DRSTZ']
  FCD cols: ['DRXFDCD', 'DRXFCSD', 'DRXFCLD']
2005-2006: food_code=['DR1IFDCD'], grams=['DR1IGRMS'], status=['DR1DRSTZ']
  FCD cols: ['DRXFDCD', 'DRXFCSD', 'DRXFCLD']
2007-2008: food_code=['DR1IFDCD'], grams=['DR1IGRMS'], status=['DR1DRSTZ']
  FCD cols: ['DRXFDCD', 'DRXFCSD', 'DRXFCLD']
2009-2010: food_code=['DR1IFDCD'], grams=['DR1IGRMS'], status=['DR1DRSTZ']
  FCD cols: ['DRXFDCD', 'DRXFCSD', 'DRXFCLD']
2011-2012: food_code=['DR1IFDCD'], grams=['DR1IGRMS'], status=['DR1DRSTZ']
  FCD cols: ['DRXFDCD', 'DRXFCSD', 'DRXFCLD']
2013-2014: food_code=['DR1IFDCD'], grams=['DR1IGRMS'], status=['DR1DRSTZ']
  FCD cols: ['DRXFDCD', 'DRXFCSD', 'DRXFCLD']
2015-2016: food_code=['DR1IFDCD'], grams

In [6]:
def harmonize_dietary(df, cycle):
    """Standardize column names across cycles."""
    rename = {}
    # Food code: DRDIFDCD (1999-2002) or DR1IFDCD (2003+)
    for c in ['DR1IFDCD', 'DRDIFDCD']:
        if c in df.columns:
            rename[c] = 'FOOD_CODE'
    # Gram weight: DRXIGRMS (1999-2002) or DR1IGRMS (2003+)
    for c in ['DR1IGRMS', 'DRXIGRMS']:
        if c in df.columns:
            rename[c] = 'FOOD_GRAMS'
    # Dietary recall status: various names
    for c in ['DR1DRSTZ', 'DRDDRSTZ', 'DRDDRSTS']:
        if c in df.columns:
            rename[c] = 'RECALL_STATUS'
    # Dietary day 1 weight (in dietary file for some cycles)
    if 'WTDRD1' in df.columns:
        pass  # keep as-is
    
    out = df.rename(columns=rename).copy()
    out['CYCLE'] = cycle
    return out

# Apply harmonization
for cycle, d in raw.items():
    d['DR1IFF'] = harmonize_dietary(d['DR1IFF'], cycle)
    print(f'{cycle}: {len(d["DR1IFF"]):,} food items, '
          f'has FOOD_CODE={"FOOD_CODE" in d["DR1IFF"].columns}, '
          f'has FOOD_GRAMS={"FOOD_GRAMS" in d["DR1IFF"].columns}')

1999-2000: 127,840 food items, has FOOD_CODE=True, has FOOD_GRAMS=True
2001-2002: 143,004 food items, has FOOD_CODE=True, has FOOD_GRAMS=True
2003-2004: 131,164 food items, has FOOD_CODE=True, has FOOD_GRAMS=True


2005-2006: 146,940 food items, has FOOD_CODE=True, has FOOD_GRAMS=True


2007-2008: 145,703 food items, has FOOD_CODE=True, has FOOD_GRAMS=True
2009-2010: 150,991 food items, has FOOD_CODE=True, has FOOD_GRAMS=True
2011-2012: 126,503 food items, has FOOD_CODE=True, has FOOD_GRAMS=True


2013-2014: 131,394 food items, has FOOD_CODE=True, has FOOD_GRAMS=True


2015-2016: 121,481 food items, has FOOD_CODE=True, has FOOD_GRAMS=True
2017-2018: 112,683 food items, has FOOD_CODE=True, has FOOD_GRAMS=True


## Build FPED legume lookup

Build a food code → V_LEGUMES (cup eq per 100g) lookup table from FPED/MPED databases.
For 1999–2002 (no FPED/MPED), use the 2003–2004 MPED as a proxy.

In [7]:
# Build per-cycle FPED lookup: food_code -> V_LEGUMES (cup eq per 100g)
fped_lookup = {}

for cycle, d in raw.items():
    if 'FPED' in d:
        fped = d['FPED']
        # Determine column names (FPED vs MPED format)
        if 'FOODCODE' in fped.columns:
            fc_col = 'FOODCODE'
            leg_col = 'V_LEGUMES (cup eq.)'
        elif 'DRDIFDCD' in fped.columns:
            fc_col = 'DRDIFDCD'
            leg_col = 'LEGUMES'  # MPED uses just LEGUMES (cup eq)
        else:
            print(f'{cycle}: unknown FPED format')
            continue
        
        # MPED may have duplicate food codes (with different modification codes)
        # Take the first/max value per food code
        lookup = fped.groupby(fc_col)[leg_col].max()
        fped_lookup[cycle] = dict(zip(lookup.index.astype(int), lookup.values))
        n_legume = sum(1 for v in lookup.values if v > 0)
        print(f'{cycle}: {len(lookup):,} food codes, {n_legume} with legumes')

# For 1999-2000 and 2001-2002, use 2003-2004 MPED as proxy
# (food codes are largely shared across early cycles)
proxy = fped_lookup.get('2003-2004', {})
for cycle in ['1999-2000', '2001-2002']:
    fped_lookup[cycle] = proxy
    print(f'{cycle}: using 2003-2004 MPED as proxy ({len(proxy):,} codes)')

2003-2004: 6,940 food codes, 235 with legumes
2005-2006: 6,921 food codes, 231 with legumes
2007-2008: 7,174 food codes, 233 with legumes
2009-2010: 7,253 food codes, 241 with legumes
2011-2012: 7,618 food codes, 278 with legumes
2013-2014: 8,536 food codes, 339 with legumes
2015-2016: 8,690 food codes, 367 with legumes
2017-2018: 7,083 food codes, 271 with legumes
1999-2000: using 2003-2004 MPED as proxy (6,940 codes)
2001-2002: using 2003-2004 MPED as proxy (6,940 codes)


In [8]:
# Classify pulse type from food descriptions and FNDDS codes
# Used only for type breakdown (beans vs chickpeas vs lentils vs peas)
# Consumption amounts come from FPED V_LEGUMES

# Build food code -> description mapping from all FCD files
fcd_frames = []
for cycle, d in raw.items():
    if 'FCD' not in d:
        continue
    fcd = d['FCD']
    code_col, desc_col = None, None
    for c in fcd.columns:
        if c == 'DRXFDCD': code_col = c
        if c == 'DRXFCLD': desc_col = c
    # DRXFMT format (1999-2002): FMTNAME, START, LABEL
    if code_col is None and 'START' in fcd.columns:
        code_col, desc_col = 'START', 'LABEL'
    if code_col and desc_col:
        sub = fcd[[code_col, desc_col]].copy()
        sub.columns = ['FOOD_CODE', 'FOOD_DESC']
        fcd_frames.append(sub)

if fcd_frames:
    food_desc_map = pd.concat(fcd_frames, ignore_index=True).drop_duplicates('FOOD_CODE')
    food_desc_map['FOOD_CODE'] = food_desc_map['FOOD_CODE'].astype(int)
    print(f'Food descriptions: {len(food_desc_map):,} codes')
else:
    food_desc_map = pd.DataFrame(columns=['FOOD_CODE', 'FOOD_DESC'])
    print('No food descriptions available')

desc_dict = dict(zip(food_desc_map['FOOD_CODE'], food_desc_map['FOOD_DESC']))

Food descriptions: 11,221 codes


In [9]:
# Classify pulse type for foods with V_LEGUMES > 0
# Uses food descriptions and FNDDS code ranges

def classify_pulse_type(food_code, desc=''):
    """Classify a legume-containing food as beans, chickpeas, lentils, or peas."""
    desc_lower = str(desc).lower()
    code = int(food_code)
    
    # Exclude soy products (tofu, soy sauce, soy milk, tempeh, edamame, miso)
    if any(kw in desc_lower for kw in ['soy', 'tofu', 'tempeh', 'edamame', 'miso']):
        return None
    
    # Chickpeas/garbanzo/hummus/falafel
    if any(kw in desc_lower for kw in ['chickpea', 'garbanzo', 'hummus', 'falafel', 'chick pea']):
        return 'chickpeas'
    
    # Lentils
    if 'lentil' in desc_lower:
        return 'lentils'
    
    # Dried peas (split peas, cowpeas, pigeon peas)
    if any(kw in desc_lower for kw in ['split pea', 'cowpea', 'pigeon pea', 'black-eyed pea',
                                         'blackeye pea']):
        return 'peas'
    
    # Bean-related keywords
    if any(kw in desc_lower for kw in [
        'bean', 'refried', 'chili con', 'chili w/', 'frijol', 'cassoulet'
    ]):
        # Exclude green beans, string beans, wax beans (vegetables, not pulses)
        if any(kw in desc_lower for kw in ['green bean', 'string bean', 'wax bean', 'snap bean']):
            return None
        return 'beans'
    
    # FNDDS code-based classification for codes without clear descriptions
    if 41000000 <= code <= 41499999:
        # Exclude soy range
        if 41500000 <= code <= 41599999:
            return None
        if 41200000 <= code <= 41299999:
            # This range includes chickpeas (41209xxx) but also bean dishes
            if 41209000 <= code <= 41209999:
                return 'chickpeas'
            return 'beans'
        if 41300000 <= code <= 41399999:
            # Peas and lentils
            if 41304000 <= code <= 41309999:
                return 'lentils'
            return 'peas'
        return 'beans'
    
    # Default: classify as beans for any food with V_LEGUMES > 0
    return 'beans'

# Identify all foods with V_LEGUMES > 0 across all cycles
all_legume_codes = set()
for cycle, lookup in fped_lookup.items():
    for code, v in lookup.items():
        if v > 0:
            all_legume_codes.add(code)

print(f'Total unique food codes with V_LEGUMES > 0: {len(all_legume_codes)}')

# Classify each
pulse_type_map = {}
for code in all_legume_codes:
    desc = desc_dict.get(code, '')
    ptype = classify_pulse_type(code, desc)
    if ptype:
        pulse_type_map[code] = ptype

print(f'Pulse food codes (excluding soy): {len(pulse_type_map)}')
type_counts = pd.Series(pulse_type_map).value_counts()
print(f'\nBy type:\n{type_counts}')

# Show samples
for pt in ['beans', 'chickpeas', 'lentils', 'peas']:
    codes = [c for c, t in pulse_type_map.items() if t == pt]
    print(f'\n  {pt.upper()} ({len(codes)} codes):')
    for c in sorted(codes)[:8]:
        print(f'    {c:>10d}  {desc_dict.get(c, "no desc")}')

Total unique food codes with V_LEGUMES > 0: 485
Pulse food codes (excluding soy): 475

By type:
beans        411
chickpeas     26
peas          22
lentils       16
Name: count, dtype: int64

  BEANS (411 codes):
      11461100  YOGURT, FROZEN, CAROB-COATED
      25210170  FRANKFURTER OR HOT DOG, CHILI-FILLED
      27111400  CHILI CON CARNE, NS AS TO BEANS
      27111405  Chili con carne with beans, from restaurant
      27111406  Chili con carne with beans, home recipe
      27111407  Chili con carne with beans, canned
      27111410  CHILI CON CARNE W/ BEANS
      27111430  CHILI CON CARNE, NS AS TO BEANS, W/ CHEESE

  CHICKPEAS (26 codes):
      41205070  HUMMUS
      41205075  Hummus, flavored
      41301990  Chickpeas, NFS
      41302000  CHICKPEAS, DRY, COOKED, NS AS TO ADDED FAT
      41302010  CHICKPEAS, DRY, COOKED, FAT ADDED
      41302011  Chickpeas, dry, cooked, made with oil
      41302012  Chickpeas, dry, cooked, made with animal fat or meat drippings
      41302013  Chick

## Build person-level pulse consumption dataset

For each food item, compute legume cup equivalents from FPED:
`legume_cups = V_LEGUMES_per100g × grams / 100`

Sum per person per day, then merge with demographics.

In [10]:
person_data = []

for cycle, d in raw.items():
    diet = d['DR1IFF'].copy()
    demo = d['DEMO'].copy()
    lookup = fped_lookup[cycle]
    
    # Filter to reliable dietary recall status (1 = reliable)
    if 'RECALL_STATUS' in diet.columns:
        diet = diet[diet['RECALL_STATUS'] == 1].copy()
    
    # Compute legume cup equivalents for each food item
    diet['V_LEGUMES_PER100'] = diet['FOOD_CODE'].apply(
        lambda c: lookup.get(int(c), 0) if pd.notna(c) else 0)
    diet['LEGUME_CUPS'] = diet['V_LEGUMES_PER100'] * diet['FOOD_GRAMS'] / 100
    
    # Classify pulse type
    diet['PULSE_TYPE'] = diet['FOOD_CODE'].apply(
        lambda c: pulse_type_map.get(int(c)) if pd.notna(c) else None)
    
    # Only count items that are classified as pulses (excludes soy)
    pulse_items = diet[(diet['LEGUME_CUPS'] > 0) & (diet['PULSE_TYPE'].notna())].copy()
    
    # Total legume cups per person
    total_cups = pulse_items.groupby('SEQN')['LEGUME_CUPS'].sum().rename('PULSE_CUPS_TOTAL')
    
    # Also keep gram weight for reference
    total_grams = pulse_items.groupby('SEQN')['FOOD_GRAMS'].sum().rename('PULSE_GRAMS_TOTAL')
    
    # By pulse type (cups)
    by_type = pulse_items.groupby(['SEQN', 'PULSE_TYPE'])['LEGUME_CUPS'].sum().unstack(fill_value=0)
    by_type.columns = [f'PULSE_CUPS_{c.upper()}' for c in by_type.columns]
    
    # Get WTDRD1 from dietary file
    wt_diet = diet.groupby('SEQN')['WTDRD1'].first() if 'WTDRD1' in diet.columns else pd.Series(dtype=float)
    
    # Filter demographics to adults 20+ with valid dietary recall
    valid_seqns = diet['SEQN'].unique()
    demo = demo[demo['SEQN'].isin(valid_seqns)].copy()
    demo = demo[demo['RIDAGEYR'] >= 20].copy()
    
    # Build person-level dataframe
    base_cols = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR',
                 'SDMVPSU', 'SDMVSTRA', 'WTMEC2YR', 'WTINT2YR']
    use_cols = [c for c in base_cols if c in demo.columns]
    person = demo[use_cols].copy()
    
    # Add dietary weight
    if len(wt_diet) > 0:
        person = person.merge(wt_diet.rename('WTDRD1'), on='SEQN', how='left')
    else:
        person['WTDRD1'] = np.nan
    
    for wc in ['WTMEC2YR', 'WTINT2YR']:
        if wc not in person.columns:
            person[wc] = np.nan
    
    person = person.merge(total_cups, on='SEQN', how='left')
    person = person.merge(total_grams, on='SEQN', how='left')
    person = person.merge(by_type, on='SEQN', how='left')
    
    person['PULSE_CUPS_TOTAL'] = person['PULSE_CUPS_TOTAL'].fillna(0)
    person['PULSE_GRAMS_TOTAL'] = person['PULSE_GRAMS_TOTAL'].fillna(0)
    for col in person.columns:
        if col.startswith('PULSE_CUPS_') and col != 'PULSE_CUPS_TOTAL':
            person[col] = person[col].fillna(0)
    
    # Ensure all pulse type columns exist
    for pt in ['BEANS', 'CHICKPEAS', 'LENTILS', 'PEAS']:
        col = f'PULSE_CUPS_{pt}'
        if col not in person.columns:
            person[col] = 0.0
    
    person['CYCLE'] = cycle
    person['IS_CONSUMER'] = (person['PULSE_CUPS_TOTAL'] > 0).astype(int)
    
    # Convert cups to oz equivalents (protein food group: 1 cup = 4 oz eq)
    person['PULSE_OZ_EQ'] = person['PULSE_CUPS_TOTAL'] * 4
    
    person_data.append(person)
    
    n_consumers = person['IS_CONSUMER'].sum()
    print(f'{cycle}: {len(person):,} adults, {n_consumers:,} consumers '
          f'({100*n_consumers/len(person):.1f}%), '
          f'mean cups={person["PULSE_CUPS_TOTAL"].mean():.3f}')

df_all = pd.concat(person_data, ignore_index=True)
print(f'\nTotal: {len(df_all):,} participants, {df_all["IS_CONSUMER"].sum():,} consumers '
      f'({100*df_all["IS_CONSUMER"].mean():.1f}%)')

1999-2000: 4,237 adults, 956 consumers (22.6%), mean cups=0.159


2001-2002: 4,744 adults, 968 consumers (20.4%), mean cups=0.142


2003-2004: 4,448 adults, 906 consumers (20.4%), mean cups=0.130


2005-2006: 4,520 adults, 881 consumers (19.5%), mean cups=0.132


2007-2008: 5,419 adults, 1,058 consumers (19.5%), mean cups=0.128


2009-2010: 5,762 adults, 1,130 consumers (19.6%), mean cups=0.136


2011-2012: 4,801 adults, 1,061 consumers (22.1%), mean cups=0.144


2013-2014: 5,047 adults, 1,066 consumers (21.1%), mean cups=0.133


2015-2016: 5,017 adults, 1,185 consumers (23.6%), mean cups=0.153


2017-2018: 4,741 adults, 963 consumers (20.3%), mean cups=0.136

Total: 48,736 participants, 10,174 consumers (20.9%)


## Create demographic variables matching the paper

In [11]:
# Age groups
df_all['AGE_GROUP'] = pd.cut(df_all['RIDAGEYR'],
                              bins=[19, 30, 50, 70, 120],
                              labels=['20-30', '31-50', '51-70', '70+'])

# Sex
df_all['SEX'] = df_all['RIAGENDR'].map({1: 'Male', 2: 'Female'})

# Race/ethnicity (RIDRETH1)
# 1=Mexican American, 2=Other Hispanic, 3=Non-Hispanic White,
# 4=Non-Hispanic Black, 5=Other Race
df_all['RACE_ETH'] = df_all['RIDRETH1'].map({
    1: 'Mexican American',
    2: 'Other',
    3: 'Non-Hispanic White',
    4: 'Non-Hispanic Black',
    5: 'Other'
})

# Education (DMDEDUC2 for adults 20+)
# 1=Less than 9th grade, 2=9-11th grade, 3=High school/GED,
# 4=Some college/AA, 5=College graduate or above
# Paper: "High school or less" vs "Some college or more"
df_all['EDUCATION'] = df_all['DMDEDUC2'].map({
    1: 'High school or less',
    2: 'High school or less',
    3: 'High school or less',
    4: 'Some college or more',
    5: 'Some college or more'
})

# Income to poverty ratio (IPR)
# Paper: <=1.85 vs >1.85
df_all['IPR_GROUP'] = pd.cut(df_all['INDFMPIR'],
                              bins=[-np.inf, 1.85, np.inf],
                              labels=['≤1.85', '>1.85'])

print('Demographic variables created')
print(f'\nAge groups:\n{df_all["AGE_GROUP"].value_counts().sort_index()}')
print(f'\nRace/ethnicity:\n{df_all["RACE_ETH"].value_counts()}')
print(f'\nEducation:\n{df_all["EDUCATION"].value_counts()}')
print(f'\nIPR:\n{df_all["IPR_GROUP"].value_counts()}')

Demographic variables created

Age groups:
AGE_GROUP
20-30     9379
31-50    16322
51-70    15214
70+       7821
Name: count, dtype: int64

Race/ethnicity:
RACE_ETH
Non-Hispanic White    21865
Non-Hispanic Black    10193
Mexican American       8556
Other                  8122
Name: count, dtype: int64

Education:
EDUCATION
High school or less     24376
Some college or more    24299
Name: count, dtype: int64

IPR:
IPR_GROUP
>1.85    24860
≤1.85    19747
Name: count, dtype: int64


## Save processed data

In [12]:
out_path = os.path.join(DERIVED, 'pulse_consumption.parquet')
df_all.to_parquet(out_path, index=False)
print(f'Saved {len(df_all):,} rows to {out_path}')
print(f'File size: {os.path.getsize(out_path)/1e6:.1f} MB')
print(f'\nColumns: {list(df_all.columns)}')

Saved 48,736 rows to /home/user/ai_assisted_us_health_data_analysis/data/derived/pulse_consumption.parquet
File size: 1.9 MB

Columns: ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'SDMVPSU', 'SDMVSTRA', 'WTMEC2YR', 'WTINT2YR', 'WTDRD1', 'PULSE_CUPS_TOTAL', 'PULSE_GRAMS_TOTAL', 'PULSE_CUPS_BEANS', 'PULSE_CUPS_CHICKPEAS', 'PULSE_CUPS_LENTILS', 'PULSE_CUPS_PEAS', 'CYCLE', 'IS_CONSUMER', 'PULSE_OZ_EQ', 'AGE_GROUP', 'SEX', 'RACE_ETH', 'EDUCATION', 'IPR_GROUP']


In [13]:
# Quick sanity check vs paper results
# Paper: 17.2% consumers, 0.39 oz eq/day overall, 2.26 oz eq/day consumers
consumers = df_all[df_all['IS_CONSUMER'] == 1]

print('=== Unweighted results (compare to paper) ===')
print(f'Total N: {len(df_all):,} (paper: 48,738)')
print(f'Consumers: {df_all["IS_CONSUMER"].sum():,} (paper: 9,186)')
print(f'Consumer %: {100*df_all["IS_CONSUMER"].mean():.1f}% (paper: 17.2%)')
print(f'\nMean pulse cups/day (all): {df_all["PULSE_CUPS_TOTAL"].mean():.3f}')
print(f'Mean pulse cups/day (consumers): {consumers["PULSE_CUPS_TOTAL"].mean():.3f}')
print(f'\nMean pulse oz eq/day (all): {df_all["PULSE_OZ_EQ"].mean():.2f} (paper: 0.39)')
print(f'Mean pulse oz eq/day (consumers): {consumers["PULSE_OZ_EQ"].mean():.2f} (paper: 2.26)')

# Also check gram weight for reference
print(f'\nMean pulse grams/day (all): {df_all["PULSE_GRAMS_TOTAL"].mean():.1f}')
print(f'Mean pulse grams/day (consumers): {consumers["PULSE_GRAMS_TOTAL"].mean():.1f}')

=== Unweighted results (compare to paper) ===
Total N: 48,736 (paper: 48,738)
Consumers: 10,174 (paper: 9,186)
Consumer %: 20.9% (paper: 17.2%)

Mean pulse cups/day (all): 0.139
Mean pulse cups/day (consumers): 0.666

Mean pulse oz eq/day (all): 0.56 (paper: 0.39)
Mean pulse oz eq/day (consumers): 2.66 (paper: 2.26)

Mean pulse grams/day (all): 46.4
Mean pulse grams/day (consumers): 222.1
